## Ejemplos de uso de REGEXP (Regular Expresions)

### Example: 3 appearances of the same letter

In [0]:
-- Databricks SQL

-- Option A:
SELECT name  
FROM hackerrank_challenges.bronze.city_c4
WHERE regexp_like(name, '(?i)a.*a.*a');

In [0]:
-- Databricks SQL

-- Option B:
-- Parameterized CTE (Clean Alternative)
WITH params AS (
    SELECT 
        'a'  AS target_char,
        '.*' AS separator,
        3    AS occurrences
)
SELECT c.name
FROM hackerrank_challenges.bronze.city_c4 c
CROSS JOIN params p
WHERE regexp_like(
    c.name, 
    concat(
        '(?i)', 
        repeat(concat(p.target_char, p.separator), p.occurrences - 1), 
        p.target_char
    )
);

In [0]:
%python
import re
import pandas as pd

city_c4_df = spark.table("hackerrank_challenges.bronze.city_c4").toPandas()

# ---------------------------------------------
# Opción A: regex directo
# ---------------------------------------------
result_a = city_c4_df.loc[
    city_c4_df["name"].str.contains("a.*a.*a", case=False, na=False),
    ["name"]
]

print(result_a)

# ---------------------------------------------
# Opción B: patrón parametrizado
# ---------------------------------------------
params = {
    "target_char": "a",
    "separator": ".*",
    "occurrences": 3,
}

char = re.escape(params["target_char"])  # protege caracteres especiales de regex
pattern = (
    "(?i)"
    + (char + params["separator"]) * (params["occurrences"] - 1)
    + char
)  # -> '(?i)a.*a.*a'

result_b = city_c4_df.loc[
    city_c4_df["name"].str.contains(pattern, regex=True, na=False),
    ["name"]
]

print(result_b)

# ---------------------------------------------
# Opción B: paquete pandas.Series.str
# ---------------------------------------------

result_c = city_c4_df.loc[city_c4_df["name"].str.lower().str.count("a") >= 3, ["name"]]

print(result_c)

### Starts and ends with vowels

In [0]:
-- Databricks SQL
WITH params AS (
    SELECT '[aeiou]' AS vowels
)
SELECT DISTINCT c.name
FROM hackerrank_challenges.bronze.city_c4 c
CROSS JOIN params p
WHERE c.name RLIKE CONCAT('(?i)^', p.vowels, '.*', p.vowels, '$');

### Ejemplos con Pandas

In [0]:
%python

import pandas as pd, re

df = pd.DataFrame({'texto': ['Pago $100', 'pago pendiente', None, 'PAGO.ok', 'Factura 2024-03']})

filtered_df = df[df['texto'].str.contains('pago', case=False, na=False)].reset_index(drop=True)

display(filtered_df)

### Buscar por las que tengan la cadena '$100'

In [0]:
%python

import pandas as pd, re

df = pd.DataFrame({'texto': ['Pago $100', 'Pago $1000', 'Retiro $100', '100', 'Pago 100', 'pago pendiente', None, 'PAGO.ok', 'Factura 2024-03', None]})

filtered_df = df[df['texto'].str.contains('$100', na=False, regex=False)].reset_index(drop=True)

print(filtered_df)